# Notebook 02: Bias Correction using Quantile Delta Mapping (QDM)
**Correcting CMIP6 precipitation bias using CHIRPS v2.0 as observational reference**

**Models**: MRI-ESM2-0, EC-Earth3, CNRM-CM6-1  
**Scenarios**: historical, SSP1-2.6, SSP2-4.5, SSP5-8.5  
**Method**: Quantile Delta Mapping (Cannon et al. 2015)  
**Calibration period**: 1981–2014 (CHIRPS v2.0)

> **Prerequisite:** Run `QDM.py prepare` then `QDM.py apply --model all --scenario all` before executing this notebook.

## 1. Setup

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as _fm

from scipy.interpolate import interp1d
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# ===== Paths ====================
PROC_DIR = Path('../../py/data/processed')       # crop_domain.py output (raw)
BC_DIR   = Path('../../py/data/bias_corrected')  # QDM.py output + CHIRPS
RESULTS  = Path('../../py/results/figures')
RESULTS.mkdir(parents=True, exist_ok=True)

# ===== CMIP6 registry ====================
MODELS = {
    'MRI-ESM2-0': {
        'ensemble':  'r1i1p1f1',
        'grid':      'gn',
        'scenarios': ['historical', 'ssp126', 'ssp245', 'ssp585'],
        'color':     '#2364a5',
    },
    'EC-Earth3': {
        'ensemble':  'r1i1p1f1',
        'grid':      'gr',
        'scenarios': ['historical', 'ssp126', 'ssp245', 'ssp585'],
        'color':     '#e07b00',
    },
    'CNRM-CM6-1': {
        'ensemble':  'r1i1p1f2',
        'grid':      'gr',
        'scenarios': ['historical', 'ssp126', 'ssp245', 'ssp585'],
        'color':     '#7b2d8b',
    },
}

ALL_SCENARIOS = ['historical', 'ssp126', 'ssp245', 'ssp585']
SCENARIO_LABELS = {
    'historical': 'Historical',
    'ssp126':     'SSP1-2.6',
    'ssp245':     'SSP2-4.5',
    'ssp585':     'SSP5-8.5',
}

CALIB_START, CALIB_END = 1981, 2014
CENTER_LAT, CENTER_LON = -6.5, 106.875
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

print('Setup complete ✔️')
print(f'Models         : {list(MODELS.keys())}')
print(f'Scenarios      : {ALL_SCENARIOS}')
print(f'Calib period   : {CALIB_START}–{CALIB_END}')
print(f'Centre cell    : lat={CENTER_LAT}, lon={CENTER_LON}')

In [ ]:
# ===== Global font settings ====================
PLOT_FONT        = 'Calibri'                     # preferred font
PLOT_FONT_ALT    = ['Arial', 'Times New Roman']  # fallback chain

_available = {f.name for f in _fm.fontManager.ttflist}
_font_to_use = PLOT_FONT if PLOT_FONT in _available else next(
    (f for f in PLOT_FONT_ALT if f in _available), 'DejaVu Sans'
)
mpl.rcParams['font.family'] = _font_to_use
mpl.rcParams['axes.titlesize']  = 11
mpl.rcParams['axes.labelsize']  = 10
mpl.rcParams['xtick.labelsize'] = 9
mpl.rcParams['ytick.labelsize'] = 9
mpl.rcParams['legend.fontsize'] = 9

print(f'Font set to: {_font_to_use}')
if _font_to_use != PLOT_FONT:
    print(f'  (Note: {PLOT_FONT!r} not found — using {_font_to_use!r} instead)')

### Helper functions

In [ ]:
def get_years(da):
    t = da.time.values
    if hasattr(t[0], 'year'):
        return np.array([v.year for v in t])
    return pd.DatetimeIndex(t).year.values


def get_months(da):
    t = da.time.values
    if hasattr(t[0], 'month'):
        return np.array([v.month for v in t])
    return pd.DatetimeIndex(t).month.values


def open_nc(fpath):
    return xr.open_dataset(
        fpath,
        decode_times=xr.coders.CFDatetimeCoder(use_cftime=True)
    )


def load_pr(fpath):
    """Open a NetCDF and return the pr DataArray, normalising units and variable name."""
    ds = open_nc(fpath)
    if 'precip' in ds and 'pr' not in ds:
        ds = ds.rename({'precip': 'pr'})
    pr = ds['pr']
    if pr.attrs.get('units', '') in ['kg m-2 s-1', 'kg/m2/s']:
        pr = pr * 86400.0
        pr.attrs['units'] = 'mm/day'
    return pr


def slice_calib(pr, start=CALIB_START, end=CALIB_END):
    yrs  = get_years(pr)
    mask = (yrs >= start) & (yrs <= end)
    return pr.isel(time=np.where(mask)[0])


def monthly_cycle(pr):
    """Return 12-element array of spatial-mean monthly climatology."""
    pr_sp  = pr.mean(dim=['lat', 'lon']) if ('lat' in pr.dims and 'lon' in pr.dims) else pr
    months = get_months(pr_sp)
    return np.array([
        float(pr_sp.isel(time=np.where(months == m)[0]).mean())
        for m in range(1, 13)
    ])


print('Helpers defined ✔️')

## 2. File Inventory

Verify that all QDM output files exist before proceeding. Expected pattern:
```
pr_day_{model}_{scenario}_{ensemble}_jakarta_qdm.nc
```

In [ ]:
# ===== CHIRPS reference ====================
chirps_file = next(BC_DIR.glob('chirps_v2_jakarta_*.nc'), None)
chirps_ok   = chirps_file is not None
print(f'CHIRPS : {"✔️ " + chirps_file.name if chirps_ok else "❌ MISSING — run QDM.py prepare"}')

# ===== QDM output files ====================
print('\nQDM output files (bias_corrected/):')
print('=' * 88)
found, missing = [], []

for model, cfg in MODELS.items():
    ensemble = cfg['ensemble']
    for scenario in cfg['scenarios']:
        fname  = f'pr_day_{model}_{scenario}_{ensemble}_jakarta_qdm.nc'
        fpath  = BC_DIR / fname
        status = '✔️' if fpath.exists() else '❌ MISSING'
        print(f'  {status}  {model:<15} {scenario:<12} {fname}')
        (found if fpath.exists() else missing).append((model, scenario))

print('=' * 88)
print(f'Found: {len(found)} / {len(found) + len(missing)}')
if missing:
    print(f'\n⚠ Missing files. Run:')
    print('  python py/bias_correction/QDM.py apply --model all --scenario all')

## 3. Load CHIRPS & Raw Historical Data

Load the CHIRPS reference and each model's raw (pre-QDM) historical file, sliced to the calibration period.

In [ ]:
if not chirps_ok:
    raise FileNotFoundError('CHIRPS file not found. Run: python py/bias_correction/QDM.py prepare')

pr_chirps     = load_pr(chirps_file)
pr_chirps_cal = slice_calib(pr_chirps)

t0_c = str(pr_chirps.time.values[0])[:10]
t1_c = str(pr_chirps.time.values[-1])[:10]
print(f'CHIRPS full period   : {t0_c} → {t1_c}  ({len(pr_chirps.time):,} days)')
print(f'CHIRPS calib period  : {len(pr_chirps_cal.time):,} days  ({CALIB_START}–{CALIB_END})')
print()

# ===== Raw historical per model ====================
pr_raw_cal = {}   # model -> calibration-period DataArray (raw)

for model, cfg in MODELS.items():
    ensemble = cfg['ensemble']
    fname    = f'pr_day_{model}_historical_{ensemble}_jakarta.nc'
    fpath    = PROC_DIR / fname
    if not fpath.exists():
        print(f'  ⚠ Raw historical file not found for {model}: {fname}')
        continue
    pr       = load_pr(fpath)
    pr_cal   = slice_calib(pr)
    pr_raw_cal[model] = pr_cal
    t0 = str(pr.time.values[0])[:10]
    t1 = str(pr.time.values[-1])[:10]
    print(f'  {model:<15}: full={t0}→{t1}  calib_days={len(pr_cal.time):,}')

## 4. Pre-Correction Bias Assessment

Compare key statistics between each raw CMIP6 model and CHIRPS over the 1981–2014 calibration period. This motivates why QDM is needed.

In [ ]:
print(f'Pre-Correction Bias')
print(f'  Calibration Period {CALIB_START}–{CALIB_END}')
print(f'    Spatial mean over Jakarta domain, centre cell stats also shown')
print('=' * 76)
print(f'  {"Source":<18} {"Mean wet [mm/d]":>16} {"WetFreq [%]":>12} {"Max [mm/d]":>11} {"SDII [mm/d]":>12}')
print('-' * 76)

chirps_mean  = float(pr_chirps_cal.where(pr_chirps_cal >= 1).mean())
chirps_wdf   = float((pr_chirps_cal >= 1).mean()) * 100
chirps_max   = float(pr_chirps_cal.max())
chirps_wdf_n = float((pr_chirps_cal >= 1).sum())
chirps_sdii  = float(pr_chirps_cal.where(pr_chirps_cal >= 1).sum()) / max(chirps_wdf_n, 1)
print(f'  {"CHIRPS v2.0":<18} {chirps_mean:>16.2f} {chirps_wdf:>12.1f} {chirps_max:>11.1f} {chirps_sdii:>12.2f}')

bias_rows = []
for model, pr_cal in pr_raw_cal.items():
    m_mean  = float(pr_cal.where(pr_cal >= 1).mean())
    m_wdf   = float((pr_cal >= 1).mean()) * 100
    m_max   = float(pr_cal.max())
    m_wdf_n = float((pr_cal >= 1).sum())
    m_sdii  = float(pr_cal.where(pr_cal >= 1).sum()) / max(m_wdf_n, 1)
    print(f'  {model:<18} {m_mean:>16.2f} {m_wdf:>12.1f} {m_max:>11.1f} {m_sdii:>12.2f}')
    print(f'  {"  bias vs CHIRPS":<18} {m_mean-chirps_mean:>+15.2f} {m_wdf-chirps_wdf:>+11.1f} {m_max-chirps_max:>+10.1f} {m_sdii-chirps_sdii:>+11.2f}')
    bias_rows.append({'Model': model, 'Mean_bias': round(m_mean - chirps_mean, 2),
                      'WDF_bias': round(m_wdf - chirps_wdf, 1), 'Max_bias': round(m_max - chirps_max, 1)})

print('=' * 76)
print('\nNote: Positive mean bias = model is too wet; negative = too dry.')
print('All CMIP6 models are expected to underestimate PRCPTOT and Rx1day')
print('over maritime SEA due to convective parameterisation (known CMIP6 issue).')

In [ ]:
# ===== Annual cycle on all models vs CHIRPS (pre-correction) ====================
fig, ax = plt.subplots(figsize=(10, 4))

chirps_cycle = monthly_cycle(pr_chirps_cal)
ax.plot(range(12), chirps_cycle, '-o', linewidth=2.5, markersize=6,
        color='black', label='CHIRPS v2.0 (obs)', zorder=5)

for model, pr_cal in pr_raw_cal.items():
    cycle = monthly_cycle(pr_cal)
    ax.plot(range(12), cycle, '--o', linewidth=1.8, markersize=4,
            color=MODELS[model]['color'], label=f'{model} (raw)', alpha=0.85)

ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('Mean Daily Precipitation [mm/day]')
ax.set_title(f'Raw CMIP6 vs CHIRPS ({CALIB_START}–{CALIB_END}) in Annual Cycle\n'
             f'Spatial mean over Jakarta domain  |  Pre-correction', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(RESULTS / '02_annual_cycle_preQDM.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. QQ-Plots Before Correction

One QQ-plot per model at the Jakarta centre cell. Deviation from the 1:1 line quantifies the distributional bias that QDM must correct.

In [ ]:
n_models  = len(pr_raw_cal)
quantiles = np.linspace(1, 99, 99)

obs_1d_center = pr_chirps_cal.sel(
    lat=CENTER_LAT, lon=CENTER_LON, method='nearest'
).values.astype(float)
obs_q = np.percentile(obs_1d_center[obs_1d_center >= 1], quantiles)

fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, (model, pr_cal) in zip(axes, pr_raw_cal.items()):
    raw_1d = pr_cal.sel(lat=CENTER_LAT, lon=CENTER_LON, method='nearest').values.astype(float)
    raw_q  = np.percentile(raw_1d[raw_1d >= 1], quantiles)

    ax.scatter(obs_q, raw_q, s=18, alpha=0.8, color=MODELS[model]['color'])
    lim = max(obs_q.max(), raw_q.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1.3, label='1:1')
    ax.set_xlabel('CHIRPS Quantiles [mm/day]')
    ax.set_ylabel(f'{model} Raw Quantiles [mm/day]')
    ax.set_title(f'{model}\nQQ-Plot Before QDM', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)

plt.suptitle('Quantile–Quantile Plots of Raw CMIP6 vs CHIRPS\nJakarta Centre Cell, Calibration Period',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / '02_qq_before_QDM.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. QDM Transfer Function (Pedagogical Demo)

Illustrates the three-step QDM logic at a single grid cell (MRI-ESM2-0, Jakarta centre):

1. **Find quantile** τ = F_hist(x_f) -> where does the future value sit in the historical CDF?
2. **Compute delta** δ = x_f / Q_hist(τ) -> multiplicative change signal relative to historical
3. **Apply to obs** x_corr = δ × Q_obs(τ) -> preserve delta while anchoring to observed distribution

Dry-day frequency is also corrected: excess wet days in the model are demoted to dry before intensity correction.

In [ ]:
DEMO_MODEL = 'MRI-ESM2-0'

if DEMO_MODEL not in pr_raw_cal:
    print(f'⚠ {DEMO_MODEL} raw historical not loaded — skipping demo.')
else:
    # ===== Centre-cell 1-D arrays ====================
    obs_1d  = pr_chirps_cal.sel(lat=CENTER_LAT, lon=CENTER_LON, method='nearest').values.astype(float)
    hist_1d = pr_raw_cal[DEMO_MODEL].sel(lat=CENTER_LAT, lon=CENTER_LON, method='nearest').values.astype(float)

    N_Q       = 100
    prob_bins = np.linspace(0, 1, N_Q + 1)

    obs_cdf  = np.quantile(obs_1d[obs_1d   >= 1], prob_bins)
    hist_cdf = np.quantile(hist_1d[hist_1d >= 1], prob_bins)

    # ===== Inline QDM (mirrors QDM.py logic exactly) ====================
    def apply_qdm_1d(future, obs_cdf, hist_cdf, prob_bins, threshold=1.0):
        corrected = future.copy().astype(float)
        wet_mask  = corrected >= threshold

        prob_obs  = (obs_1d  >= threshold).mean()
        prob_hist = (hist_1d >= threshold).mean()

        # Step 1: dry-day frequency correction
        if prob_hist > prob_obs and wet_mask.sum() > 0:
            ratio       = prob_obs / max(prob_hist, 1e-9)
            wet_idx     = np.where(wet_mask)[0]
            n_to_dry    = int(len(wet_idx) * (1 - ratio))
            if n_to_dry > 0:
                demote            = wet_idx[np.argsort(corrected[wet_idx])][:n_to_dry]
                corrected[demote] = 0.0
                wet_mask[demote]  = False

        # Step 2: QDM intensity correction
        if wet_mask.sum() > 0:
            x_f       = corrected[wet_mask]
            f_tau     = interp1d(hist_cdf, prob_bins, bounds_error=False,
                                 fill_value=(prob_bins[0], prob_bins[-1]))
            f_obs_q   = interp1d(prob_bins, obs_cdf,  bounds_error=False,
                                 fill_value=(obs_cdf[0],  obs_cdf[-1]))
            f_hist_q  = interp1d(prob_bins, hist_cdf, bounds_error=False,
                                 fill_value=(hist_cdf[0], hist_cdf[-1]))
            tau            = f_tau(x_f)
            x_hist_tau     = f_hist_q(tau)
            x_obs_tau      = f_obs_q(tau)
            delta          = np.clip(x_f / np.maximum(x_hist_tau, 1e-6), 0, 5.0)
            corrected[wet_mask] = np.maximum(delta * x_obs_tau, threshold)

        corrected[~wet_mask] = 0.0
        return np.maximum(corrected, 0.0)

    hist_corrected_1d = apply_qdm_1d(hist_1d, obs_cdf, hist_cdf, prob_bins)

    # ===== QQ-plot: before vs after ====================
    quantiles = np.linspace(1, 99, 99)
    obs_q     = np.percentile(obs_1d[obs_1d >= 1], quantiles)
    raw_q     = np.percentile(hist_1d[hist_1d >= 1], quantiles)
    corr_q    = np.percentile(hist_corrected_1d[hist_corrected_1d >= 1], quantiles)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, model_q, label, color in [
        (axes[0], raw_q,  'Before QDM', '#D73027'),
        (axes[1], corr_q, 'After QDM',  '#4DAF4A'),
    ]:
        ax.scatter(obs_q, model_q, s=20, alpha=0.85, color=color, label=label)
        lim = max(obs_q.max(), model_q.max()) * 1.05
        ax.plot([0, lim], [0, lim], 'k--', linewidth=1.3, label='1:1')
        ax.set_xlabel('CHIRPS Quantiles [mm/day]')
        ax.set_ylabel(f'Model Quantiles [mm/day]')
        ax.set_title(f'{label}', fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.4)

    plt.suptitle(f'QDM Transfer Function Demo\n'
                 f'{DEMO_MODEL}, Jakarta Centre Cell Calibration period {CALIB_START}–{CALIB_END}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS / f'02_qq_qdm_demo_{DEMO_MODEL}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ===== Wet-day frequency check ====================
    print(f'\nWet-day frequency at centre cell ({DEMO_MODEL}):')
    print(f'  CHIRPS (obs)  : {(obs_1d  >= 1).mean()*100:.1f}%')
    print(f'  Raw model     : {(hist_1d >= 1).mean()*100:.1f}%')
    print(f'  After QDM     : {(hist_corrected_1d >= 1).mean()*100:.1f}%')
    print(f'\nMean wet-day intensity at centre cell:')
    print(f'  CHIRPS (obs)  : {obs_1d[obs_1d >= 1].mean():.2f} mm/day')
    print(f'  Raw model     : {hist_1d[hist_1d >= 1].mean():.2f} mm/day')
    print(f'  After QDM     : {hist_corrected_1d[hist_corrected_1d >= 1].mean():.2f} mm/day')

## 7. Post-Correction Validation

Load the QDM-corrected historical files and compare against CHIRPS.  
Annual cycle and QQ-plots should now align closely with the observed distribution.

In [ ]:
pr_qdm_cal = {}   # model -> calibration-period QDM DataArray

for model, cfg in MODELS.items():
    ensemble = cfg['ensemble']
    fname    = f'pr_day_{model}_historical_{ensemble}_jakarta_qdm.nc'
    fpath    = BC_DIR / fname
    if not fpath.exists():
        print(f'  ⚠ QDM output not found for {model}: {fname}')
        continue
    pr              = load_pr(fpath)
    pr_qdm_cal[model] = slice_calib(pr)
    print(f'  ✔️ Loaded: {fname}  ({len(pr_qdm_cal[model].time):,} calibration days)')

In [ ]:
# ===== Post-correction statistics ====================
print(f'Post-QDM Statistics — Calibration Period {CALIB_START}–{CALIB_END}')
print('=' * 72)
print(f'  {"Source":<20} {"Mean wet [mm/d]":>15} {"WetFreq [%]":>12} {"Max [mm/d]":>11}')
print('-' * 72)

chirps_mean = float(pr_chirps_cal.where(pr_chirps_cal >= 1).mean())
chirps_wdf  = float((pr_chirps_cal >= 1).mean()) * 100
chirps_max  = float(pr_chirps_cal.max())
print(f'  {"CHIRPS v2.0":<20} {chirps_mean:>15.2f} {chirps_wdf:>12.1f} {chirps_max:>11.1f}')

for model in pr_qdm_cal:
    pr_c    = pr_qdm_cal[model]
    m_mean  = float(pr_c.where(pr_c >= 1).mean())
    m_wdf   = float((pr_c >= 1).mean()) * 100
    m_max   = float(pr_c.max())
    print(f'  {model + " (QDM)":<20} {m_mean:>15.2f} {m_wdf:>12.1f} {m_max:>11.1f}')
    print(f'  {"  residual bias":<20} {m_mean-chirps_mean:>+14.2f} {m_wdf-chirps_wdf:>+11.1f} {m_max-chirps_max:>+10.1f}')

print('=' * 72)
print('\nNote: All "bc" weights in ensemble_weights.json are ~0.333 by construction')
print('because QDM forces CDF convergence during calibration, so this is expected.')

In [ ]:
# ===== Annual cycle of all models before and after QDMs ====================
fig, axes = plt.subplots(1, len(pr_qdm_cal), figsize=(6 * len(pr_qdm_cal), 4), sharey=True)
if len(pr_qdm_cal) == 1:
    axes = [axes]

chirps_cycle = monthly_cycle(pr_chirps_cal)

for ax, model in zip(axes, pr_qdm_cal):
    ax.plot(range(12), chirps_cycle, '-o', linewidth=2.5, markersize=6,
            color='black', label='CHIRPS (obs)', zorder=5)

    if model in pr_raw_cal:
        raw_cycle = monthly_cycle(pr_raw_cal[model])
        ax.plot(range(12), raw_cycle, '--o', linewidth=1.6, markersize=4,
                color=MODELS[model]['color'], alpha=0.5, label=f'{model} (raw)')

    qdm_cycle = monthly_cycle(pr_qdm_cal[model])
    ax.plot(range(12), qdm_cycle, '-o', linewidth=2.0, markersize=5,
            color=MODELS[model]['color'], label=f'{model} (QDM)')

    ax.set_xticks(range(12))
    ax.set_xticklabels(MONTH_NAMES, fontsize=8)
    ax.set_title(model, fontweight='bold')
    ax.grid(True, alpha=0.4)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Mean Daily Precipitation [mm/day]')
plt.suptitle(f'Before & After QDM vs CHIRPS ({CALIB_START}–{CALIB_END}) in Annual Cycle\n'
             f'Spatial mean over Jakarta domain',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / '02_annual_cycle_postQDM.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== QQ-plots after QDM — all models ====================
quantiles     = np.linspace(1, 99, 99)
obs_1d_center = pr_chirps_cal.sel(lat=CENTER_LAT, lon=CENTER_LON, method='nearest').values.astype(float)
obs_q         = np.percentile(obs_1d_center[obs_1d_center >= 1], quantiles)

n_models = len(pr_qdm_cal)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, model in zip(axes, pr_qdm_cal):
    qdm_1d = pr_qdm_cal[model].sel(
        lat=CENTER_LAT, lon=CENTER_LON, method='nearest'
    ).values.astype(float)
    qdm_q = np.percentile(qdm_1d[qdm_1d >= 1], quantiles)

    ax.scatter(obs_q, qdm_q, s=18, alpha=0.85, color=MODELS[model]['color'])
    lim = max(obs_q.max(), qdm_q.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1.3, label='1:1')
    ax.set_xlabel('CHIRPS Quantiles [mm/day]')
    ax.set_ylabel('QDM-Corrected Quantiles [mm/day]')
    ax.set_title(f'{model}\nQQ-Plot After QDM', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)

plt.suptitle(f'Quantile–Quantile Plots of QDM-Corrected CMIP6 vs CHIRPS\n'
             f'Jakarta Centre Cell, Calibration Period',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / '02_qq_after_QDM.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Output File Verification on All Scenarios

Confirm period coverage of all QDM output files (historical + all SSP scenarios).

In [ ]:
print('QDM Output Verification — Period Coverage')
print('=' * 80)
print(f'  {"Model":<15} {"Scenario":<12} {"Start":>4} {"End":>10} {"Days":>13} {"Mean [mm/d]":>17}')
print('-' * 80)

for model, cfg in MODELS.items():
    ensemble = cfg['ensemble']
    for scenario in cfg['scenarios']:
        fname = f'pr_day_{model}_{scenario}_{ensemble}_jakarta_qdm.nc'
        fpath = BC_DIR / fname
        if not fpath.exists():
            print(f'  {model:<15} {scenario:<12} {"MISSING":>32}')
            continue
        pr = load_pr(fpath)
        t0   = str(pr.time.values[0])[:10]
        t1   = str(pr.time.values[-1])[:10]
        n    = len(pr.time)
        mean = float(pr.mean())
        print(f'  {model:<15} {SCENARIO_LABELS.get(scenario, scenario):<12} {t0:>10} {t1:>12} {n:>8,} {mean:>8.2f}')
        pr.close()

print('=' * 80)
print('\nAll files verified. Proceed to: 03_precipitation_indices.ipynb')

## ✅ Summary

**QDM methodology:**
- Quantile Delta Mapping (Cannon et al. 2015) corrects both distributional bias and preserves the future climate change signal
- Dry-day frequency bias is corrected first (lightest wet days demoted to dry)
- Intensity correction uses a multiplicative delta: δ = x_future / Q_hist(τ), applied to the observed CDF at the same quantile τ
- Calibration: 1981–2014 against CHIRPS v2.0; applied spatially to every grid cell

**Key findings from raw data:**
- All three models show good agreement with CHIRPS annual cycle after QDM
- QQ-plots confirm distributional convergence in the calibration period
- Post-QDM ensemble weights (`bc` key in `ensemble_weights.json`) are all ≈0.333, which is expected because QDM forces CDF convergence; use `raw` weights for downstream weighting

**Files produced by `QDM.py`:**
```
py/data/bias_corrected/
  chirps_v2_jakarta_1981_2014.nc
  pr_day_{model}_{scenario}_{ensemble}_jakarta_qdm.nc   (12 files: 3 models × 4 scenarios)
```

**Next:** `03_precipitation_indices.ipynb`